# 🛡️ अल्टीमेट क्लाउड एंटीवायरस और प्राइवेसी सैंडबॉक्स

In [ ]:
import os, sys, shutil, glob
from IPython.display import FileLink, display

print('🔧 Sab tools pre-installed hain (ClamAV, unzip, unrar, 7z, mediafire-dl)')
print('✅ Ready.\n')

# 1. Link input
print('🔒 Privacy active.')
mf_link = input('➡️ MediaFire ya file link paste karo aur Enter dabao: ')

# 2. Download
print('\n⏳ Downloading...')
# Purane archives hatao
for f in glob.glob('*.zip') + glob.glob('*.rar') + glob.glob('*.7z'):
    try:
        os.remove(f)
    except:
        pass
os.system(f'mediafire-dl "{mf_link}"')

# 3. Archive dhundho (hidden files ignore)
archive_files = glob.glob('*.zip') + glob.glob('*.rar') + glob.glob('*.7z')
archive_files = [f for f in archive_files if not os.path.basename(f).startswith('.')]

if not archive_files:
    print('❌ Koi archive file (.zip, .rar, .7z) download nahi hui.')
    print('   Link galat ho sakta hai ya file public nahi hai.')
    sys.exit(1)

archive_file = max(archive_files, key=lambda f: os.path.getmtime(f))
print(f'📥 Downloaded archive: {archive_file}')
print(f'📏 Size: {os.path.getsize(archive_file)} bytes')

# --- File type aur first bytes check ---
print('\n🔍 File type analysis:')
!file "{archive_file}"
print('\n🔍 First 100 bytes (hex):')
!xxd -l 100 "{archive_file}"

# ZIP file PK signature check (\x50\x4B\x03\x04)
with open(archive_file, 'rb') as f:
    header = f.read(4)
if header != b'PK\x03\x04':
    print(f'❌ File ZIP signature nahi hai. Header: {header}')
    print('   Download shayad corrupt hai ya file asli ZIP nahi hai.')
    sys.exit(1)

# --- Archive contents list ---
ext = archive_file.split('.')[-1].lower()
print(f'\n📂 Contents of {archive_file}:')
if ext == 'zip':
    !unzip -l "{archive_file}"
elif ext == 'rar':
    !unrar l "{archive_file}"
elif ext == '7z':
    !7z l "{archive_file}"

# 4. Extract
print('\n⏳ Extracting...')
if os.path.exists('extracted_files'):
    shutil.rmtree('extracted_files')
os.makedirs('extracted_files', exist_ok=True)

if ext == 'zip':
    !unzip -o "{archive_file}" -d extracted_files/
elif ext == 'rar':
    !unrar x -o+ "{archive_file}" extracted_files/
elif ext == '7z':
    !7z x "{archive_file}" -oextracted_files/ -aoa
else:
    !unzip -o "{archive_file}" -d extracted_files/ 2>/dev/null
    if not os.listdir('extracted_files'):
        print('❌ Extraction fail.')
        sys.exit(1)
print('✅ Extraction complete.\n')

# 5. Scan with ClamAV
print('🛡️ Scanning...')
!clamscan --version
scan_result = !clamscan -r --remove extracted_files/

virus_found = False
for line in scan_result:
    if 'Infected files:' in line:
        print(line.strip())
        if 'Infected files: 0' not in line:
            virus_found = True

print('\n--- Scan Summary ---')
for line in scan_result[-10:]:
    print(line)

# 6. Result & Download
print('\n' + '='*50)
if virus_found:
    print('❌ Virus mila! File server se delete kar di gayi.')
    print('🛑 Download blocked!')
else:
    print('✅ File 100% safe hai. Clean zip bana rahe hain...')
    shutil.make_archive('safe_download', 'zip', 'extracted_files')
    print('📥 Safe download ready:')
    display(FileLink('safe_download.zip'))
print('='*50)